# Homework — Part A: Weight Detective

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/homework/weight-space/HW-Part-A-Weight-Detective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

## Background

In November 2026, the blog post [*Sparks of Weight-Space Programming*](https://rikiyat.github.io/blog/weight-space-programming/) showed a frontier LLM ("GPT-6 Astra") reading the weights of small neural networks as if they were programs. In its first experiment, four policy networks had been trained to play four grid games (Breakout, Pong, Pac-Man, Freeway). Astra was given **only the weights**, plus a description of the inputs and outputs, and named the game of every network correctly, mainly by looking at:

* where each input channel's first-layer weights are concentrated (for Breakout, the agent's weights lie along the bottom row: the paddle), and
* which actions the output layer actually distinguishes.

**Your job is to do the same with the tools from this course.** You may use an LLM to help you write code or to discuss ideas, but every conclusion in your report must be supported by a figure or a number that you produced.

## The networks

Every network has the same architecture as in the paper: `500 → 32 → 32 → 5` with `tanh`.

**Input**: 5 channels of a 10×10 grid (flattened to 500 numbers, channel-major: `x.view(5, 10, 10)`):

| Channel | Meaning |
|---|---|
| 0 | the agent (the thing the policy controls) |
| 1 | moving objects |
| 2 | reward targets |
| 3 | obstacles |
| 4 | the previous frame (agent + moving objects one step ago) |

**Output**: 5 actions: `0 no-op, 1 up, 2 down, 3 left, 4 right`.

Each network was trained to imitate an expert player of one game on this grid. The candidate games include **Breakout, Freeway and Pac-Man**, but some networks may have played **games that are not on this list**, and not every network is necessarily a clean single-game player.

**Rules**
* **Level 1 (required)**: weights only. Do not run the networks on any input.
* **Level 2**: you may feed inputs that you design yourself to the networks, to test a hypothesis from Level 1. You do not have a game simulator, and you may not write one to play the games.

In [ ]:
import os, urllib.request
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/homework/weight-space/data/"
NAMES = ["example_pong"] + list("ABCDEFG")
os.makedirs("data", exist_ok=True)
for n in NAMES:
    fn = f"data/policy_{n}.pt"
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE + f"policy_{n}.pt", fn)

def policy():
    return nn.Sequential(nn.Linear(500, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 5))

nets = {}
for n in NAMES:
    net = policy()
    net.load_state_dict(torch.load(f"data/policy_{n}.pt"))
    nets[n] = net.eval()
CHANNELS = ["agent", "moving", "reward", "obstacle", "prev frame"]
ACTIONS = ["no-op", "up", "down", "left", "right"]
print({k: v.shape for k, v in nets["A"].state_dict().items()})

## Worked example: a Pong network

`policy_example_pong.pt` was trained on **Pong**: the agent is a paddle 3 cells tall on the **right** column, the opponent's paddle (an obstacle) is on the **left** column, and the ball (a moving object) bounces between them. We walk through the analysis on this network; you will repeat it on networks A–G.

### Step 1: draw the first layer back onto the grid

The first layer `W1` has shape `(32, 500)`. Row `j` is what hidden unit `j` "looks at": reshape it to `(5, 10, 10)` and each channel becomes a 10×10 picture. Summing `|w|` over the 32 units shows where each channel is used at all.

In [ ]:
def channel_maps(net):
    W1 = net[0].weight.detach().view(32, 5, 10, 10)
    return W1.abs().sum(0)                       # (5, 10, 10)

def show_channel_maps(name):
    maps = channel_maps(nets[name])
    fig, axes = plt.subplots(1, 5, figsize=(13, 2.8))
    for c, ax in enumerate(axes):
        ax.imshow(maps[c], cmap="viridis")
        ax.set_title(CHANNELS[c]); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"policy {name}: sum over hidden units of |first-layer weight|")
    plt.show()

show_channel_maps("example_pong")

What we see for Pong:
* **agent**: only the right-most column is used: the paddle lives there and moves up and down.
* **obstacle**: only the left-most column: the opponent's paddle.
* **moving** / **prev frame**: spread over the playing field (the ball can be anywhere), with nothing on the left and right edge columns where the paddles are.
* **reward**: looks like **random noise**. In Pong nothing is ever drawn in the reward channel.

📌 **Question 1.** Why would the weights of a channel that is *never used* look like noise rather than zero? (Hint: what were the weights before training, and what is the gradient of a weight whose input is always 0? Check your answer by creating an untrained `policy()` and drawing its maps.)

### Step 2: individual hidden units

The sum hides the details. Plot the weights (with sign) of a few hidden units with the largest weights on the agent channel.

In [ ]:
def show_units(name, channel=0, k=4):
    W1 = nets[name][0].weight.detach().view(32, 5, 10, 10)
    top = W1[:, channel].abs().sum((1, 2)).argsort(descending=True)[:k]
    fig, axes = plt.subplots(k, 5, figsize=(11, 2.2 * k))
    for r, j in enumerate(top.tolist()):
        m = W1[j].abs().max()
        for c in range(5):
            axes[r, c].imshow(W1[j, c], cmap="RdBu_r", vmin=-m, vmax=m)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0: axes[r, c].set_title(CHANNELS[c])
        axes[r, 0].set_ylabel(f"unit {j}")
    plt.show()

show_units("example_pong")

Red = positive weight, blue = negative. A unit that is positive where the paddle is high and negative where the ball is low compares the two positions: exactly the information a Pong player needs ("is the ball above or below my paddle?").

### Step 3: the output layer

Which actions does the network use? Look at the output biases and at the size of each action's output weights.

In [ ]:
def show_output(name):
    W3, b3 = nets[name][4].weight.detach(), nets[name][4].bias.detach()
    fig, axes = plt.subplots(1, 2, figsize=(9, 2.5))
    axes[0].bar(ACTIONS, b3); axes[0].set_title("output bias")
    axes[1].bar(ACTIONS, W3.norm(dim=1)); axes[1].set_title("output weight norm")
    fig.suptitle(f"policy {name}"); plt.show()

show_output("example_pong")

In Pong only `no-op`, `up` and `down` make sense. The actions the expert never chose get a **low bias** (the network learned to rarely output them). The weight norms are a weaker clue: unused actions still have weights of a similar size.

📌 **Question 2.** Why are biases a better clue than weight norms for "this action is never used"?

### Step 4: trace a path from input to action

Near zero, `tanh(z) ≈ z`, so the whole network is approximately linear: `logits ≈ W3 · W2 · W1 · x`. The matrix `M = W3 @ W2 @ W1` (shape `5 × 500`) says how much each input cell pushes each action. This is only an approximation (the units saturate), but it often shows the network's main logic.

In [ ]:
def show_linear_path(name, channels=(0, 1)):
    net = nets[name]
    M = (net[4].weight @ net[2].weight @ net[0].weight).detach().view(5, 5, 10, 10)
    fig, axes = plt.subplots(len(channels), 5, figsize=(12, 2.4 * len(channels)), squeeze=False)
    for r, c in enumerate(channels):
        for a in range(5):
            m = M[:, c].abs().max()
            axes[r, a].imshow(M[a, c], cmap="RdBu_r", vmin=-m, vmax=m)
            axes[r, a].set_xticks([]); axes[r, a].set_yticks([])
            if r == 0: axes[r, a].set_title(ACTIONS[a])
        axes[r, 0].set_ylabel(CHANNELS[c])
    plt.show()

show_linear_path("example_pong")

📌 **Question 3.** In the `moving` row, is the "up" action pushed by a ball in the upper or in the lower part of the field? Does that match how you would play Pong?

### Step 5 (Level 2): test a hypothesis with a designed input

Put the paddle in the middle of the right column and the ball near the top: the network should move **up**.

In [ ]:
def make_input(agent=(), moving=(), reward=(), obstacle=(), prev=()):
    x = torch.zeros(5, 10, 10)
    for c, cells in enumerate([agent, moving, reward, obstacle, prev]):
        for (r, col) in cells:
            x[c, r, col] = 1
    return x.view(1, 500)

x = make_input(agent=[(4, 9), (5, 9), (6, 9)], moving=[(1, 6)], prev=[(4, 9), (5, 9), (6, 9), (2, 5)], obstacle=[(4, 0), (5, 0), (6, 0)])
with torch.no_grad():
    p = nets["example_pong"](x).softmax(-1)[0]
print({a: round(v, 3) for a, v in zip(ACTIONS, p.tolist())})

## Your task: networks A–G

For each network, find out which game it played. Some hints of what to look out for:

* Two different games may put the agent in the same place. What do the **other** channels tell you then?
* A network may have been trained on **more than one** game. What would its maps look like?
* A network may not have learned anything at all. How would you recognise that, and how sure can you be?
* For a game that is not on the candidate list, describe the game you think it is: where is the agent, how does it move, what are the moving objects, what does it collect, what does it avoid?

In [ ]:
for n in "ABCDEFG":
    show_channel_maps(n)

In [ ]:
for n in "ABCDEFG":
    show_output(n)

📌 **Question 4.** Fill in a table: network → game (or description) → at least **two independent clues**, each with the figure or number it comes from → how confident you are and why.

📌 **Question 5.** Which of your clues were strong and which were weak? Was there a network where the clues disagreed?

📌 **Question 6 (Level 2, optional).** For two networks, design inputs that test your hypothesis (as in Step 5) and report the results. Did any result surprise you?

📌 **Question 7 (discussion).** The paper's networks were trained with reinforcement learning (PPO), ours by imitating an expert. Which of the clues you used would you expect to still work for PPO-trained networks, and which might not?